# **DIDP model and dual bound declaration**

In [ ]:
%%writefile TSPTW_DIDP_model_and_dual_bound_declaration.py
import sys
import os
import numpy as np
import modified_didppy as m_dp
from ortools.linear_solver import pywraplp
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.optimize import linear_sum_assignment
from numpy.linalg import eigh
from functools import lru_cache

# --- GLOBALS ---
current_num_locations = 0
current_travel_cost = []
current_avail_time = []
current_due_date = []

# --- READER ---
def read_tsptw_format(file_path):
    with open(file_path, "r") as f:
        lines = f.readlines()
    
    all_tokens = []
    for line in lines:
        all_tokens.extend(line.strip().split())
    iterator = iter(all_tokens)
    
    try:
        n = int(next(iterator))
        c = []
        for i in range(n):
            row = []
            for j in range(n):
                row.append(float(next(iterator)))
            c.append(row)
        avail = []
        due = []
        for i in range(n):
            avail.append(float(next(iterator)))
            due.append(float(next(iterator)))
        return n, c, avail, due
    except StopIteration:
        return 0, [], [], []

# --- MODEL DEFINITION (EXACT USER SPECIFICATION) ---
def creation_of_didp_model_function():
    # Expects global variables: num_locations, dist_matrix, time_windows
    num_locations = current_num_locations
    travel_cost = current_travel_cost
    avail_time = current_avail_time
    due_date = current_due_date
    
    # 1. Setup Model
    model = m_dp.Model(float_cost=True)
    customer = model.add_object_type(number=num_locations)

    # 2. State Variables
    # unvisited: Set of customers to visit (excluding depot 0)
    unvisited = model.add_set_var(object_type=customer, target=list(range(1, num_locations)))
    # location: Current node
    location = model.add_element_var(object_type=customer, target=0)
    # time: Current cumulative time (resource)
    curr_time = model.add_float_resource_var(target=0.0, less_is_better=True)

    # 3. Data Tables & Helpers
    travel_time_table = model.add_float_table(travel_cost)
    
    # 4. Transitions: Visit Customer j
    for j in range(1, num_locations):
        visit = m_dp.Transition(
            name="visit {}".format(j),
            cost=travel_time_table[location, j] + m_dp.FloatExpr.state_cost(),
            preconditions=[
                unvisited.contains(j),
                # Feasibility check: Must arrive at j by its Due Date
                # Note: We can arrive early and wait, so we check if arrival <= due_date
                curr_time + travel_time_table[location, j] <= due_date[j]
            ],
            effects=[
                (unvisited, unvisited.remove(j)),
                (location, j),
                # Time update: max(arrival_time, ready_time)
                # arrival_time = curr_time + travel_time
                (curr_time, m_dp.max(curr_time + travel_time_table[location, j], avail_time[j])),
            ],
        )
        model.add_transition(visit)

    # 5. Transition: Return to Depot (0)
    return_to_depot = m_dp.Transition(
        name="return",
        cost=travel_time_table[location, 0] + m_dp.FloatExpr.state_cost(),
        effects=[
            (location, 0),
            (curr_time, curr_time + travel_time_table[location, 0]),
        ],
        preconditions=[
            unvisited.is_empty(), 
            location != 0,
        ],
    )
    model.add_transition(return_to_depot)

    # 6. Base Case
    model.add_base_case([unvisited.is_empty(), location == 0])

    for j in range(1, num_locations):
        model.add_state_constr(
            ~unvisited.contains(j) | (curr_time + travel_time_table[location, j] <= due_date[j])
        )

    # 8. Bundle
    metadata = {
        "num_locations": num_locations,
        "distance_matrix": travel_cost,
        "avail_time": avail_time,
        "due_date": due_date,
        "unvisited_var": unvisited,
        "location_var": location,
        "time_var": curr_time
    }
    
    didp_bundle = (model, metadata)
    return didp_bundle

# ==========================================
# DUAL BOUNDS (ALL INCLUDED)
# ==========================================

# 1. Persistent 3-Index LP Relaxation
def create_persistent_lp_relaxation_3_index_dual_bounds(metadata):
    n_nodes = metadata['num_locations']
    unvisited_set_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    dist_matrix = metadata['distance_matrix']
    
    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver: return lambda state: 0.0

    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    u = {i: solver.NumVar(0, n_nodes, f'u_{i}') for i in range(n_nodes)}

    cons_out, cons_in = {}, {}
    for i in range(n_nodes):
        c_out = solver.Constraint(0, 0, f'deg_out_{i}')
        c_in = solver.Constraint(0, 0, f'deg_in_{i}')
        for j in range(n_nodes):
            if i != j: 
                c_out.SetCoefficient(x[(i, j)], 1)
                c_in.SetCoefficient(x[(j, i)], 1)
        cons_out[i] = c_out
        cons_in[i] = c_in

    infinity = solver.infinity()
    for i in range(n_nodes):
        if i == 0: continue
        for j in range(n_nodes):
            if j == 0 or i == j: continue
            c_mtz = solver.Constraint(-infinity, n_nodes - 1, f'mtz_{i}_{j}')
            c_mtz.SetCoefficient(u[i], 1)
            c_mtz.SetCoefficient(u[j], -1)
            c_mtz.SetCoefficient(x[(i, j)], n_nodes)

    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    @lru_cache(maxsize=10000)
    def _solve_3idx(active_tuple):
        current_node = active_tuple[0]
        active_set = set(active_tuple[1:])
        active_set.add(current_node); active_set.add(0) 
        max_u = len(active_set)
        for i in range(n_nodes):
            if i in active_set:
                if i == current_node:
                    cons_out[i].SetBounds(1, 1); cons_in[i].SetBounds(0, 0); u[i].SetBounds(0, 0)
                elif i == 0:
                    cons_out[i].SetBounds(0, 0); cons_in[i].SetBounds(1, 1); u[i].SetBounds(0, max_u)
                else:
                    cons_out[i].SetBounds(1, 1); cons_in[i].SetBounds(1, 1); u[i].SetBounds(0, max_u)
            else:
                cons_out[i].SetBounds(0, 0); cons_in[i].SetBounds(0, 0); u[i].SetBounds(0, 0)
        solver.SetTimeLimit(100)
        if solver.Solve() in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
            return float(objective.Value())
        return 0.0

    def h_lp_relaxation_3_idx(state):
        unvisited = state[unvisited_set_var]
        current_node = state[location_var]
        if not unvisited and current_node == 0: return 0.0
        key = (current_node,) + tuple(sorted(list(unvisited)))
        return _solve_3idx(key)

    return h_lp_relaxation_3_idx

# 2. Persistent 2-Index LP Relaxation
def create_persistent_lp_relaxation_2_index_dual_bounds(metadata):
    n_nodes = metadata['num_locations']
    dist_matrix = metadata['distance_matrix']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver: return lambda state: 0.0
    infinity = solver.infinity()

    x = {}
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')
    u = {i: solver.NumVar(0, n_nodes, f'u_{i}') for i in range(1, n_nodes)}

    cons_deg_out, cons_deg_in = {}, {}
    for i in range(n_nodes):
        c_out = solver.Constraint(0, 0, f'deg_out_{i}')
        c_in = solver.Constraint(0, 0, f'deg_in_{i}')
        for j in range(n_nodes):
            if i != j: 
                c_out.SetCoefficient(x[(i, j)], 1)
                c_in.SetCoefficient(x[(j, i)], 1)
        cons_deg_out[i] = c_out
        cons_deg_in[i] = c_in

    for i in range(1, n_nodes):
        for j in range(1, n_nodes):
            if i != j:
                c = solver.Constraint(-infinity, n_nodes - 1, f'mtz_{i}_{j}')
                c.SetCoefficient(u[j], 1); c.SetCoefficient(u[i], -1); c.SetCoefficient(x[(i, j)], n_nodes)

    objective = solver.Objective()
    for i in range(n_nodes):
        for j in range(n_nodes):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    @lru_cache(maxsize=10000)
    def _solve_2idx(active_tuple):
        current_loc = active_tuple[0]
        active_set = set(active_tuple[1:])
        active_set.add(current_loc); active_set.add(0)
        max_u = len(active_set)
        for i in range(n_nodes):
            if i in active_set:
                cons_deg_out[i].SetBounds(1, 1); cons_deg_in[i].SetBounds(1, 1)
                if i > 0:
                    if i == current_loc: u[i].SetBounds(0, 0)
                    else: u[i].SetBounds(0, max_u)
            else:
                cons_deg_out[i].SetBounds(0, 0); cons_deg_in[i].SetBounds(0, 0)
                if i > 0: u[i].SetBounds(0, 0)
        solver.SetTimeLimit(100)
        if solver.Solve() in [pywraplp.Solver.OPTIMAL, pywraplp.Solver.FEASIBLE]:
            return float(objective.Value())
        return 0.0

    def h_lp_relaxation_2_idx(state):
        unvisited = state[unvisited_var]
        current_loc = state[location_var]
        if not unvisited and current_loc == 0: return 0.0
        key = (current_loc,) + tuple(sorted(list(unvisited)))
        return _solve_2idx(key)

    return h_lp_relaxation_2_idx

# 3. Persistent TSPTW Relaxed Model (Big-M)
def create_persistent_tsptw_lp_bound(metadata):
    num_locations = metadata['num_locations']
    dist_matrix = metadata['distance_matrix']
    avail_time = metadata['avail_time']
    due_date = metadata['due_date']
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']
    time_var = metadata['time_var']
    
    big_m = {}
    for i in range(num_locations):
        for j in range(num_locations):
            if i != j: big_m[(i, j)] = 10**6 #max(due_date[i] + dist_matrix[i][j] - avail_time[j], 0.0)

    solver = pywraplp.Solver.CreateSolver('GLOP')
    if not solver: return lambda state: 0.0
    infinity = solver.infinity()

    x = {}; w = {}
    for i in range(num_locations):
        w[i] = solver.NumVar(avail_time[i], due_date[i], f'w_{i}')
        for j in range(num_locations):
            if i != j: x[(i, j)] = solver.NumVar(0, 1, f'x_{i}_{j}')

    cons_flow = {}
    for i in range(num_locations):
        c = solver.Constraint(0, 0, f'flow_{i}')
        for j in range(num_locations):
            if i != j:
                c.SetCoefficient(x[(i, j)], 1)
                c.SetCoefficient(x[(j, i)], -1)
        cons_flow[i] = c

    for i in range(num_locations):
        for j in range(num_locations):
            if i != j:
                M = big_m[(i, j)]
                if M > 0:
                    c = solver.Constraint(-infinity, M - dist_matrix[i][j], f'time_{i}_{j}')
                    c.SetCoefficient(w[i], 1); c.SetCoefficient(w[j], -1); c.SetCoefficient(x[(i, j)], M)

    objective = solver.Objective()
    for i in range(num_locations):
        for j in range(num_locations):
            if i != j: objective.SetCoefficient(x[(i, j)], dist_matrix[i][j])
    objective.SetMinimization()

    @lru_cache(maxsize=10000)
    def _solve_tsptw(input_tuple):
        current_node = input_tuple[0]
        current_time = input_tuple[1]
        active_set = set(input_tuple[2:]); active_set.add(current_node); active_set.add(0)

        for i in range(num_locations):
            if i in active_set:
                if i == current_node:
                    cons_flow[i].SetBounds(1, 1)
                    lb = max(avail_time[i], current_time)
                    ub = due_date[i]
                    if lb > ub: return float('inf')
                    w[i].SetBounds(lb, ub)
                elif i == 0:
                    cons_flow[i].SetBounds(-1, -1)
                    w[i].SetBounds(avail_time[i], due_date[i])
                else:
                    cons_flow[i].SetBounds(0, 0)
                    w[i].SetBounds(avail_time[i], due_date[i])
                
                for j in range(num_locations):
                    if i != j:
                        if j in active_set: x[(i, j)].SetBounds(0, 1)
                        else: x[(i, j)].SetBounds(0, 0)
            else:
                cons_flow[i].SetBounds(0, 0)
                w[i].SetBounds(avail_time[i], due_date[i])
                for j in range(num_locations):
                    if i != j: x[(i, j)].SetBounds(0, 0)

        solver.SetTimeLimit(100)
        status = solver.Solve()
        if status == pywraplp.Solver.OPTIMAL: return float(objective.Value())
        elif status == pywraplp.Solver.INFEASIBLE: return float('inf')
        return 0.0

    def h_lp_tsptw(state):
        unvisited = state[unvisited_var]
        current_node = state[location_var]
        current_time = state[time_var]
        if not unvisited and current_node == 0: return 0.0
        return _solve_tsptw((current_node, round(current_time, 2)) + tuple(sorted(list(unvisited))))

    return h_lp_tsptw

# --- REGISTRY ---
def dual_bound_expression_function(didp_bundle):
    model, metadata = didp_bundle
    cost_matrix = np.array(metadata['distance_matrix'])
    min_outgoing_arr = np.min(cost_matrix + np.diag([np.inf]*len(cost_matrix)), axis=1)
    min_incoming_arr = np.min(cost_matrix + np.diag([np.inf]*len(cost_matrix)), axis=0)
    
    unvisited_var = metadata['unvisited_var']
    location_var = metadata['location_var']

    # LP Bounds
    h_lp_3idx = create_persistent_lp_relaxation_3_index_dual_bounds(metadata)
    h_lp_2idx = create_persistent_lp_relaxation_2_index_dual_bounds(metadata)
    h_lp_tsptw = create_persistent_tsptw_lp_bound(metadata)

    # Combinatorial Bounds
    @lru_cache(maxsize=100000)
    def _calc_degree(active_tuple):
        nodes = list(active_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)] + np.diag([np.inf]*len(nodes))
        return float(0.5 * (np.sum(np.min(sub_mat, axis=0)[1:]) + np.sum(np.min(sub_mat, axis=1)[:-1])))

    def h_degree_average(state):
        U = state[unvisited_var]; curr = state[location_var]
        if not U and curr == 0: return 0.0
        active_list = [curr] + sorted(list(U))
        if 0 not in active_list: active_list.append(0)
        return _calc_degree(tuple(active_list))

    @lru_cache(maxsize=100000)
    def _calc_min_flow_static(unvisited_tuple):
        return sum(min_outgoing_arr[u] for u in unvisited_tuple), sum(min_incoming_arr[u] for u in unvisited_tuple)

    def h_global_min_flow(state):
        U = state[unvisited_var]; curr = state[location_var]
        if not U and curr == 0: return 0.0
        val_out, val_in = _calc_min_flow_static(tuple(sorted(list(U))))
        if curr != 0: val_out += min_outgoing_arr[curr]; val_in += min_incoming_arr[0]
        return float(max(val_out, val_in))

    @lru_cache(maxsize=100000)
    def _calc_mst(unvisited_tuple):
        if not unvisited_tuple: return 0.0
        nodes = [0] + list(unvisited_tuple)
        return float(minimum_spanning_tree(cost_matrix[np.ix_(nodes, nodes)]).sum())

    def h_mst(state):
        return _calc_mst(tuple(sorted(list(state[unvisited_var]))))

    @lru_cache(maxsize=100000)
    def _calc_1tree(unvisited_tuple):
        subset = list(unvisited_tuple)
        depot_edges = sorted(cost_matrix[0, subset])
        if len(subset) > 1: mst_val = minimum_spanning_tree(cost_matrix[np.ix_(subset, subset)]).sum()
        else: mst_val = 0.0
        return float(mst_val + depot_edges[0] + (depot_edges[1] if len(depot_edges)>1 else 0.0))

    def h_1tree(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_1tree(tuple(sorted(list(U))))

    @lru_cache(maxsize=100000)
    def _calc_assignment(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple)
        sub_mat = cost_matrix[np.ix_(nodes, nodes)] + np.diag([np.inf]*len(nodes))
        r, c = linear_sum_assignment(sub_mat)
        return float(sub_mat[r, c].sum())

    def h_assignment(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_assignment(tuple(sorted(list(U))))

    @lru_cache(maxsize=100000)
    def _calc_eigen(unvisited_tuple):
        nodes = [0] + list(unvisited_tuple); N = len(nodes)
        if N < 2: return 0.0
        D_sub = cost_matrix[np.ix_(nodes, nodes)]
        one = np.ones((N, 1)); P = np.eye(N) - (one @ one.T) / N
        try: eigvals = np.flip(eigh((-P @ D_sub @ P + (-P @ D_sub @ P).T)/2)[0])
        except: return 0.0
        eigvals = eigvals[np.abs(eigvals) > 1e-9]
        coeffs = np.array([1 - np.cos(2 * np.pi * k / N) for k in range(1, N)])
        phi = 0.0
        if N > 1:
            num = (N - 1) // 2 if N % 2 == 1 else N // 2 - 1
            if 2*num <= len(eigvals) and num <= len(coeffs):
                phi = sum(coeffs[k-1] * (eigvals[2*k-2] + eigvals[2*k-1]) for k in range(1, num+1))
            if N % 2 == 0 and N > 1 and N-2 < len(eigvals): phi += 2 * eigvals[N-2]
        return float(phi)

    def h_eigen(state):
        U = state[unvisited_var]
        if not U: return 0.0
        return _calc_eigen(tuple(sorted(list(U))))

    from evolutionary_algorithm_lib.utils import automatic_creation_of_dual_bounds_registry
    return automatic_creation_of_dual_bounds_registry(locals())

Overwriting tsptw_domain.py


# ** Execution worker**

In [ ]:
%%writefile worker_tsptw_time_limit.py
import sys
import os
import ast
import json
import time
import threading

# --- CONFIGURATION ---
MEMORY_LIMIT_MB = 4000 

# --- PATHS ---
PROJECT_ROOT = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP"
LIB_PATH = os.path.join(PROJECT_ROOT, "Evolutionary_algorithm")

sys.path.append(os.getcwd())
if PROJECT_ROOT not in sys.path: sys.path.append(PROJECT_ROOT)
if LIB_PATH not in sys.path: sys.path.append(LIB_PATH)

try:
    import psutil
    import modified_didppy as m_dp
    import TSPTW_DIDP_model_and_dual_bound_declaration # Uses your new domain
    from evolutionary_algorithm_lib import combining_modified_didppy_solver_with_chromosome
except ImportError as e:
    print(json.dumps({"status": "error", "message": f"Import Error: {e}"}))
    sys.exit(1)

# --- GUARD ---
def memory_guard():
    process = psutil.Process(os.getpid())
    limit_bytes = MEMORY_LIMIT_MB * 1024 * 1024
    while True:
        try:
            if process.memory_info().rss > limit_bytes:
                sys.stderr.write(f"\n[Guard] Memory limit reached.\n")
                sys.stderr.flush()
                os._exit(1)
            time.sleep(1)
        except: break

# --- MAIN ---
if __name__ == "__main__":
    try:
        threading.Thread(target=memory_guard, daemon=True).start()

        if len(sys.argv) < 5: raise ValueError("Args mismatch")
        
        instance_name = sys.argv[1]
        chromosome_str = sys.argv[2]
        data_dir = sys.argv[3]
        time_limit = float(sys.argv[4])

        # Load TSPTW Data
        file_path = os.path.join(data_dir, instance_name)
        n, c, r, d = TSPTW_DIDP_model_and_dual_bound_declaration.read_tsptw_format(file_path)
        TSPTW_DIDP_model_and_dual_bound_declaration.current_num_locations = n
        TSPTW_DIDP_model_and_dual_bound_declaration.current_travel_cost = c
        TSPTW_DIDP_model_and_dual_bound_declaration.current_avail_time = r
        TSPTW_DIDP_model_and_dual_bound_declaration.current_due_date = d

        start_time = time.time()
        
        # Standard Execution
        result = combining_modified_didppy_solver_with_chromosome(
            chromosome=ast.literal_eval(chromosome_str),
            didp_model_registry=TSPTW_DIDP_model_and_dual_bound_declaration.creation_of_didp_model_function,
            dual_bound_expression_function=TSPTW_DIDP_model_and_dual_bound_declaration.dual_bound_expression_function,
            solver_time_limit=time_limit,
            output_other_result=True,
            print_timing_stats=True,
            solver_quite=False 
        )
        duration = time.time() - start_time

        if result is None:
            output = {"status": "timeout", "cost": float('inf'), "duration": duration}
        elif isinstance(result, (float, int)): 
             output = {"status": "success", "cost": float(result), "duration": duration}
        else: 
            cost, is_opt, gen, exp, stats = result
            output = {
                "status": "success",
                "cost": cost,
                "is_optimal": is_opt,
                "generated": gen,
                "expanded": exp,
                "duration": duration,
                "stats": stats
            }
        
        print(json.dumps(output))

    except Exception as e:
        print(json.dumps({"status": "error", "message": str(e)}))
        sys.exit(1)

Overwriting worker_tsptw.py


# **Execution manager**

In [ ]:
import subprocess
import pandas as pd
import os
import sys
import json
import re

# ==========================================
# CONFIGURATION
# ==========================================
SOLVER_TIME_LIMIT = 1800 # 30 Minutes
ENABLE_BATCH_MODE = True
SINGLE_TARGET_INSTANCE = "88.txt"
n_50_signal = True

# Paths for TSPTW
if n_50_signal:
    DATA_DIR = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\2_TSPTW_dual_bounds_and_models\Datasets\n50"
    INPUT_CSV_PATH = "result_of_ea_TSPTW_dual_bounds_50_cus.csv"
    VERIFICATION_OUTPUT_CSV = "TSPTW_EA_dual_bound_verification_results_50_cus.csv"
    LOG_DIR = "TSPTW_EA_dual_bound_solver_logs_n50"
else:
    DATA_DIR = r"C:\Users\ACER\Desktop\Code\0_Thesis_implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\2_TSPTW_dual_bounds_and_models\Datasets\n20"
    INPUT_CSV_PATH = "result_of_ea_TSPTW_dual_bounds_20_cus.csv"
    VERIFICATION_OUTPUT_CSV = "TSPTW_EA_dual_bound_verification_results_20_cus.csv"
    LOG_DIR = "TSPTW_EA_dual_bound_solver_logs_n20"

os.makedirs(LOG_DIR, exist_ok=True)

def run_tsptw_verification():
    print(f"🔹 Starting TSPTW Verification (Limit: {SOLVER_TIME_LIMIT}s)...")
    
    if not os.path.exists(INPUT_CSV_PATH):
        print("❌ CSV missing.")
        return
    
    df_input = pd.read_csv(INPUT_CSV_PATH)
    targets = []
    
    if ENABLE_BATCH_MODE:
        processed = set()
        if os.path.exists(VERIFICATION_OUTPUT_CSV):
            try: processed = set(pd.read_csv(VERIFICATION_OUTPUT_CSV)['Instance'].values)
            except: pass
        for _, row in df_input.iterrows():
            if row['Instance'] not in processed:
                targets.append(row)
    else:
        row = df_input[df_input['Instance'] == SINGLE_TARGET_INSTANCE]
        if not row.empty: targets.append(row.iloc[0])

    print(f"📝 Queued {len(targets)} instances.")

    # --- MAIN LOOP ---
    for i, row in enumerate(targets):
        instance = row['Instance']
        chrom_str = row['Best_Chromosome']
        print(f"\n[{i+1}/{len(targets)}] Processing {instance}...")

        safe_name = instance.replace(".txt", "")
        log_file = os.path.join(LOG_DIR, f"solver_log_{safe_name}.txt")
        
        # Call the TSPTW Worker
        cmd = [
            sys.executable, "-u", "worker_tsptw_time_limit.py", 
            instance, chrom_str, DATA_DIR, str(SOLVER_TIME_LIMIT)
        ]

        try:
            with open(log_file, "wb") as f:
                process = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT)
            
            # --- Parse Logs for Recovery ---
            best_cost = None
            nodes_expanded = 0
            try:
                with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
                    for line in f:
                        line = line.strip()
                        match_bound = re.search(r"New primal bound:\s+(\d+(\.\d+)?)", line)
                        if match_bound: best_cost = float(match_bound.group(1))
                        match_exp = re.search(r"expanded:\s+(\d+)", line)
                        if match_exp: nodes_expanded = int(match_exp.group(1))
            except: pass

            output_data = {}
            if process.returncode != 0:
                print(f"⚠️ Worker Died (Code {process.returncode}). Using scraped data.")
                obj_val = f"{best_cost} (Crashed)" if best_cost else "OOM/Error"
                output_data = {
                    "Instance": instance,
                    "Objective Value": obj_val,
                    "Optimality": False,
                    "Nodes Expanded": nodes_expanded,
                    "Nodes Generated": "Crash",
                    "Total Times (s)": "Crash",
                    "Dual Bound Chromosome": chrom_str  # <--- Added
                }
            else:
                try:
                    with open(log_file, "r", encoding="utf-8", errors="ignore") as f:
                        lines = f.readlines()
                    json_str = "{}"
                    for line in reversed(lines):
                        if line.strip().startswith("{"):
                            json_str = line.strip()
                            break
                    res = json.loads(json_str)
                    
                    if res.get('status') == 'success':
                        print(f"   -> Success. Cost: {res['cost']}")
                        output_data = {
                            "Instance": instance,
                            "Objective Value": res['cost'],
                            "Optimality": res.get('is_optimal', False),
                            "Nodes Expanded": res.get('expanded', 0),
                            "Nodes Generated": res.get('generated', 0),
                            "Total Times (s)": res['duration'],
                            "Bridging Time (s)": res.get('stats', {}).get('total_bridge_time', 0),
                            "Python Bound Time (s)": res.get('stats', {}).get('python_dual_bound_calc_time', 0),
                            "Dual Bound Chromosome": chrom_str # <--- Added
                        }
                    elif res.get('status') == 'timeout':
                        print(f"   ⚠️ Timeout.")
                        final_cost = res.get('cost')
                        if final_cost == float('inf') and best_cost is not None:
                            final_cost = f"{best_cost} (Timeout)"
                        output_data = {
                            "Instance": instance, 
                            "Objective Value": final_cost,
                            "Optimality": False, 
                            "Nodes Expanded": nodes_expanded,
                            "Total Times (s)": SOLVER_TIME_LIMIT,
                            "Dual Bound Chromosome": chrom_str # <--- Added
                        }
                    else:
                        print(f"   ⚠️ Worker Error: {res.get('message')}")
                        output_data = {
                            "Instance": instance, 
                            "Objective Value": "Worker Error",
                            "Dual Bound Chromosome": chrom_str # <--- Added
                        }
                except:
                     output_data = {
                        "Instance": instance,
                        "Objective Value": f"{best_cost} (JSON Error)" if best_cost else "Error",
                        "Nodes Expanded": nodes_expanded,
                        "Dual Bound Chromosome": chrom_str # <--- Added
                     }

            df_new = pd.DataFrame([output_data])
            if os.path.exists(VERIFICATION_OUTPUT_CSV):
                df_existing = pd.read_csv(VERIFICATION_OUTPUT_CSV)
                if instance in df_existing['Instance'].values:
                    df_existing = df_existing[df_existing['Instance'] != instance]
                pd.concat([df_existing, df_new], ignore_index=True).to_csv(VERIFICATION_OUTPUT_CSV, index=False)
            else:
                df_new.to_csv(VERIFICATION_OUTPUT_CSV, index=False)
            print("   ✅ Saved.")

        except Exception as e:
            print(f"   ❌ Execution Failed: {e}")

    print("\n✅ Done.")

if __name__ == "__main__":
    run_tsptw_verification()

🔹 Starting TSPTW Verification (Limit: 1800s)...
📝 Queued 20 instances.

[1/20] Processing 85.txt...
   -> Success. Cost: 1664.0
   ✅ Saved.

[2/20] Processing 12.txt...
   -> Success. Cost: 1551.0
   ✅ Saved.

[3/20] Processing 14.txt...
   -> Success. Cost: 1723.0
   ✅ Saved.

[4/20] Processing 72.txt...
   -> Success. Cost: 1718.0
   ✅ Saved.

[5/20] Processing 46.txt...
   -> Success. Cost: 1667.0
   ✅ Saved.

[6/20] Processing 3.txt...
   -> Success. Cost: 1628.0
   ✅ Saved.

[7/20] Processing 64.txt...
   -> Success. Cost: 1798.0
   ✅ Saved.

[8/20] Processing 56.txt...
   -> Success. Cost: 1641.0
   ✅ Saved.

[9/20] Processing 84.txt...
   -> Success. Cost: 1440.0
   ✅ Saved.

[10/20] Processing 27.txt...
   -> Success. Cost: 1502.0
   ✅ Saved.

[11/20] Processing 10.txt...
   -> Success. Cost: 1541.0
   ✅ Saved.

[12/20] Processing 31.txt...
   -> Success. Cost: 1494.0
   ✅ Saved.

[13/20] Processing 59.txt...
   -> Success. Cost: 1418.0
   ✅ Saved.

[14/20] Processing 80.txt...